## Section F - http.csv

In [7]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import TimestampType
from pyspark.sql.window import Window

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CERT Analysis") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.executorEnv.SPARK_LOCAL_IP", "127.0.0.1").config("spark.driver.memory", "4g").config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()


In [3]:
# utils.py contains all helpers
from helpers import code_quality, parse_date, letters_cleaning

In [2]:
http_raw = spark.read.csv(
    "../data/http.csv",
    header=True,
    inferSchema=True
).drop("content") 

In [3]:
http_raw.printSchema()
http_raw.show(3, truncate=False)

root
 |-- id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- user: string (nullable = true)
 |-- pc: string (nullable = true)
 |-- url: string (nullable = true)

+------------------------+-------------------+-------+-------+---------------------------------------------------------------------------------------------------------------------------------+
|id                      |date               |user   |pc     |url                                                                                                                              |
+------------------------+-------------------+-------+-------+---------------------------------------------------------------------------------------------------------------------------------+
|{W1D0-Y6WN34LF-1843MHXW}|01/02/2010 06:35:31|LRG0155|PC-0450|http://groupon.com/1980_eruption_of_Mount_St_Helens/ronnholm/vagryyrpghnycebcreglculfvpfpnyraqneovyyvneqfgevpxfubgf481322958.aspx|
|{H6X2-S7KB89GP-5832OBFJ}|01/02/2010 06:38:35|L

Contains: HTTP traffic logs - visited URL, time, user, computer. <br>
Columns: ID, date, user, computer, URL, content <br>
Significance for GNN: visits to job portals, WikiLeaks, Tor, etc. = red flags <br>
NOTE: To apply a file, Spark will stream it - it does not load the entire file into RAM. The operation can take up to a minute.

In [4]:
http_raw.cache()

DataFrame[id: string, date: string, user: string, pc: string, url: string]

In [5]:
def spark_quality_check(df):
    print("Rows:", df.count())
    
    print("\nMissing values per column:")
    df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]).show()
    
    print("\nDuplicates:")
    print(df.count() - df.dropDuplicates().count())

In [8]:
spark_quality_check(http_raw)

Rows: 29553383

Missing values per column:
+---+----+----+---+---+
| id|date|user| pc|url|
+---+----+----+---+---+
|  0|   0|   0|  0|  0|
+---+----+----+---+---+


Duplicates:
0


In [9]:
# cleaning
http = http_raw \
    .dropDuplicates(["id"])\
    .filter(F.col("user").isNotNull()) \
    .filter(F.col("url").isNotNull())

http_raw.unpersist()  # just after cleaning we immediately clean our cache

DataFrame[id: string, date: string, user: string, pc: string, url: string]

In [11]:
# utils.py contains all helpers
from helpers import code_quality, parse_date, letters_cleaning

In [12]:
http = parse_date(http, "date")

In [13]:
http.show(3)

+--------------------+-------------------+-------+-------+--------------------+-------------------+----+-----+---+----+---------------+-----------------+
|                  id|               date|   user|     pc|                 url|          timestamp|year|month|day|hour|day_of_the_week|not_working_hours|
+--------------------+-------------------+-------+-------+--------------------+-------------------+----+-----+---+----+---------------+-----------------+
|{A0A0-A5YA20MD-39...|08/09/2010 17:08:41|MNF0048|PC-3076|http://hubpages.c...|2010-08-09 17:08:41|2010|    8|  9|  17|              2|            false|
|{A0A0-C2MV35EH-89...|08/17/2010 09:40:02|CWM0042|PC-4939|http://wordpress....|2010-08-17 09:40:02|2010|    8| 17|   9|              3|            false|
|{A0A0-D3BJ06LM-33...|11/04/2010 13:15:29|SIS0806|PC-5622|http://gotjumped....|2010-11-04 13:15:29|2010|   11|  4|  13|              5|            false|
+--------------------+-------------------+-------+-------+------------------

In [14]:
# We are taking out the domain from URL — we are only interested in site's pattern, not whole site names
# ex: "http://www.wikileaks.org/..." → "wikileaks.org"
http = http.withColumn(
    "domain",
    F.regexp_extract(F.col("url"), r"https?://(?:www\.)?([^/]+)", 1)
)

In [15]:
sus_domains = [
    "wikileaks.org",
    "thepiratebay.org",
    "torproject.org",
    "indeed.com",
    "monster.com",
    "careerbuilder.com",
    "linkedin.com",
    "dropbox.com",
    "wetransfer.com",
    "pastebin.com"
] # this list is from CERT literature

In [16]:
http = http.withColumn(
    "sus_domain",
    F.col("domain").isin(sus_domains)
)

In [18]:
print("Top 20 most visited domains:")
http.groupBy("domain") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20)

Top 20 most visited domains:
+----------------+------+
|          domain| count|
+----------------+------+
|         nfl.com|696955|
|    linkedin.com|556789|
|       inbox.com|497792|
|   mediafire.com|482238|
|      hilton.com|461215|
|     eonline.com|456559|
|  shareasale.com|446324|
|    outbrain.com|433182|
|  wellsfargo.com|432113|
|    theblaze.com|422769|
|     conduit.com|377079|
|  vistaprint.com|371304|
|        imdb.com|354686|
|digitalpoint.com|304673|
|    sidereel.com|295236|
|         npr.org|280737|
| officedepot.com|279234|
|   fatwallet.com|274350|
|      fiverr.com|267158|
|      sfgate.com|266468|
+----------------+------+
only showing top 20 rows


In [19]:
print("Users that visited sus websites most frequently:")
http.filter(F.col("sus_domain")) \
    .groupBy("user") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(15)

Users that visited sus websites most frequently:
+-------+-----+
|   user|count|
+-------+-----+
|DMR0116| 9241|
|TAC0118| 7956|
|DSR0614| 7810|
|IBP0601| 7235|
|IXH0619| 7019|
|OPP0624| 7011|
|WMH0615| 6861|
|CCN0611| 6424|
|MKK0616| 6420|
|JDH0115| 6411|
|VHS0606| 6390|
|MLB0010| 6386|
|RRP0117| 6130|
|ECM0004| 5869|
|LRK0423| 5859|
+-------+-----+
only showing top 15 rows


In [20]:
# feature engineering for gnn
http_features = http \
    .groupBy("user", "year", "month", "day") \
    .agg(
        F.count("*").alias("visits_amount"),
        F.countDistinct("domain").alias("different_domains"),
        F.sum(F.col("sus_domain").cast("int")).alias("sus_domains"),
        F.sum(F.col("not_working_hours").cast("int")).alias("nightly_visits")
    )

In [ ]:
import pandas as pd
import os


chunks = []
chunk_size = 10000

for i, row in enumerate(http_features.toLocalIterator()):
    chunks.append(row.asDict())
    
    if (i + 1) % chunk_size == 0:
        part = (i + 1) // chunk_size
        pd.DataFrame(chunks).to_parquet(
            "your_link",
            index=False
        )
        chunks = []
        print(f"saved {i+1} rows")

if chunks:
    pd.DataFrame(chunks).to_parquet(
        "your_link",
        index=False
    )

print("chunks saved")

saved 10000 rows
saved 20000 rows
saved 30000 rows
saved 40000 rows
saved 50000 rows
saved 60000 rows
saved 70000 rows
saved 80000 rows
saved 90000 rows
saved 100000 rows
saved 110000 rows
saved 120000 rows
saved 130000 rows
saved 140000 rows
saved 150000 rows
saved 160000 rows
saved 170000 rows
saved 180000 rows
saved 190000 rows
saved 200000 rows
saved 210000 rows
saved 220000 rows
saved 230000 rows
saved 240000 rows
saved 250000 rows
saved 260000 rows
saved 270000 rows
saved 280000 rows
saved 290000 rows
saved 300000 rows
saved 310000 rows
saved 320000 rows
saved 330000 rows
saved 340000 rows
chunks saved
